<a href="https://colab.research.google.com/github/em2tech/Machine_Learning_SJU/blob/main/HW3/Analysis_HW3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Part II - Analysis**

1.**What challenges did you encounter when loading and inspecting a raw external dataset from Kaggle compared to pre-packaged datasets, specifically regarding implicit data types like whitespace entries in TotalCharges?**

**Answer:** Like in previous assignment, raw Kaggle CSV are not pre-packaged datasets like those in sklearn which come already cleaned where in numeric columns are numeric, the target is encoded, and missing values are clearly marked. As a result, every assumption has to be checked before modeling with the raw data.
The biggest challenge was TotalCharges. It contains dollar amounts, yet df.info() reported it as a text dtype rather than a number, and describe() left it out of the numeric summary. The cause was 11 rows containing a single space instead of a value. Because a space is a valid string, df.isna().sum() reported zero missing values across all 7,043 rows. A quick missing-value check would have declared the data clean when it was not.
If this had gone unnoticed, two things could have happened: the column could have been treated as categorical and one-hot encoded into thousands of meaningless columns (one per unique dollar amount), or StandardScaler would have failed on string values.
Investigating the 11 rows showed that every one had tenure = 0. These are brand-new customers who had not yet received a bill, so the blanks have a logical explanation rather than being random errors. Other raw-data issues included a text target (Yes/No) that had to be mapped to 1/0, SeniorCitizen stored as 0/1 while all other demographic fields were text, and customerID, a unique value for all 7,043 rows that carries no predictive information.


**2. How did you determine which columns required missing value imputation versus other data cleaning methods, and what strategy did you choose?**

**Answer:** : Rather than applying one cleaning method to every column, I looked at the specific issue in each column and chose a fix that matched it.

•	Unique identifiers: customerID contained 7,043 distinct values across 7,043 rows, so every customer had a different ID. A column like this offers no pattern for the model to learn, only a way to memorize individual rows, so I removed it instead of imputing anything.

•	Incorrect data types: TotalCharges was stored as text because some entries were blank spaces. Before any imputation could happen, I had to replace those blanks with NaN and convert the column to a numeric type.

•	True missing values: once the conversion was done, TotalCharges turned out to be the only column with missing data, affecting 11 rows, or 0.16% of the dataset. Every other column was fully populated.

Since so few rows were affected, removing them would have been a valid option. I decided to impute instead so that no customers were lost and so the pipeline could handle missing values if they appear in new data later. For the numeric columns, I filled missing values with the median.


For the numeric columns, I filled missing values with the median. TotalCharges has a strong right skew: the median is 1,397, but the mean is 2,283 because a group of long term customers have totals as high as $8,685.
Those high values pull the mean upward, so the median gives a better picture of a typical customer. For the categorical columns, I included most_frequent imputation as a precaution. None of them had missing values in this dataset, but adding it means the pipeline won't fail if future data contains gaps.

Those high values pull the mean upward, so the median gives a better picture of a typical customer. For the categorical columns, I included most_frequent imputation as a precaution. None of them had missing values in this dataset, but adding it means the pipeline won't fail if future data contains gaps.

It's also worth considering an approach based on what the data represents. All 11 customers with missing TotalCharges had a tenure of 0, meaning they were new and hadn't been billed yet. Filling their totals with 0 would arguably be more accurate than using the median of 1,397, which overstates what they had actually been charged. Because only 11 out of 7,043 rows were involved, this decision makes almost no difference to the model's results. Still, it illustrates how investigating the reason behind missing data can lead to a more accurate fix than relying on a statistical default alone.

**3. Explain how the Scikit-Learn ColumnTransformer and Pipeline streamline the data preparation process and prevent data leakage across preprocessing and training steps.**

**Answer:** ColumnTransformer sends different groups of columns through different preprocessing steps and then combines the results into one feature matrix. Here, the 3 numeric columns went through median imputation and scaling, while the 16 categorical columns went through most-frequent imputation and one-hot encoding, producing 46 model features. Doing this by hand would mean transforming each group separately, tracking column order, and stitching arrays back together, which is repetitive and error-prone.

Pipeline chains the preprocessing and the classifier into a single object. One call to .fit() fits every step in order, and .predict() or .predict_proba() automatically applies the exact same transformations to new data. This made Task 7 simple: the entire workflow was rebuilt and refit for three splits and two scalers (six models) inside one loop, without duplicating code. It also allowed feature names to be recovered with get_feature_names_out() for the coefficient analysis in Task 8.

The most important benefit is preventing data leakage. Several steps learn values from data: the imputer learns the median ($1,397 on the full data), the scaler learns means and standard deviations or minimums and maximums, and the encoder learns the list of categories. If these were fit on the full dataset before splitting, information from the test set would influence training, and the evaluation scores would be overly optimistic. Because the pipeline is fit only on X_train, all learned values come from the training data, and the test set is transformed using those same values, exactly as brand-new customers would be in production. OneHotEncoder(handle_unknown='ignore') also means an unseen category at prediction time will not crash the model.


**4. How did comparing StandardScaler versus MinMaxScaler impact your model training, convergence, or final evaluation metrics?**

**Answer**: Both scalers were tested on all three splits using the same random seed;
StandardScaler was slightly better in every split, but the margin was very small: at most 0.004 in accuracy, 0.009 in churn F1, and 0.003 in ROC-AUC. Convergence was identical, with each pair of models needing the same number of epochs (27 to 32). In practice, both scalers produced equivalent models.

The reason both work well is that scaling solves the main problem for SGDClassifier: it uses a single learning rate for all features. Unscaled, TotalCharges (range $18.80 to $8,684.80) would dominate tenure (0 to 72 months) and the 0/1 one-hot columns, causing unstable weight updates and slow or failed convergence. Either scaler puts the numeric features on a comparable scale.

StandardScaler's small edge is consistent with theory. It centers features at 0, which works well with gradient descent and the L2 penalty, which pulls weights toward 0. MinMaxScaler maps features to [0, 1], matching the range of the one-hot columns, but it is sensitive to outliers because the most extreme value defines the range. Since these three numeric features have bounded ranges with no extreme outliers, MinMaxScaler did not suffer much, which explains the near-identical results.


**5. How does configuring SGDClassifier(loss='log_loss') differ from standard linear classification models, and what advantages do probability outputs and ROC-AUC metrics provide when evaluating customer churn?**

**Answers:** SGDClassifier(loss='log_loss') fits the same model as logistic regression, a weighted sum of features passed through a sigmoid to estimate churn probability, but it learns differently. LogisticRegression uses batch solvers that process the full dataset at each step, while SGD updates weights one sample at a time. Both reached nearly identical results on the 80/20 split (ROC-AUC 0.842 vs. 0.840). SGD scales to large datasets, supports online learning with partial_fit, and can switch loss functions easily, but it is more sensitive to scaling and hyperparameters: with the default alpha=le-4, coefficients were unstable and churn F1 was only 0.52, while alpha=le-3 stabilized them and raised F1 to 0.60.

Probability outputs let the business rank customers by churn risk and adjust the decision threshold. At the default 0.5 threshold, the model caught 215 of 374 churners (57% recall) and flagged 123 loyal customers by mistake; lowering the threshold would catch more churners at the cost of more false alarms, depending on the cost of a retention offer versus a lost customer.

ROC-AUC matters because the data is imbalanced (26.5% churners), so predicting "stays" for everyone would already score 73.5% accuracy. ROC-AUC measures ranking ability across all thresholds, and scores of 0.840–0.846 mean the model ranks a real churner above a non-churner about 84% of the time.

Results were stable across the 80/20, 70/30, and 60/40 splits, with ROC-AUC varying by only 0.005 and accuracy by less than one percentage point, suggesting the model learned the main patterns even with 60% of the data for training.

**6. Based on your model coefficients or feature evaluation, which features show the strongest relationship with customer churn, and what operational business insights do these findings reveal?**

**Answers:** The 80/20 model's coefficients, confirmed by raw churn rates, show that tenure is the strongest predictor of retention (−1.08), with 47.4% of first-year customers churning compared with 17.1% of everyone else, while month-to-month contracts are the strongest driver of churn (+0.76, 42.7% churn vs. 2.8% for two-year contracts), followed by fiber optic internet (+0.58, 41.9% vs. 19.0% for DSL), electronic check payments (+0.36, about three times the churn of automatic payments), and the absence of tech support or online security (+0.33 and +0.32). These patterns suggest the company should encourage longer contracts, strengthen onboarding and loyalty efforts during the first year, investigate pricing or reliability issues with fiber service, offer incentives to switch to autopay, bundle support services, and use the model's churn probabilities to target retention offers at the highest-risk customers rather than offering blanket discounts. However, these coefficients reflect association rather than causation, and correlated features such as tenure and TotalCharges partly offset each other, so the insights should be tested through retention experiments before guiding major business decisions.